# UniContext — Python quickstart

`unicontext` is a Cython extension over the UniContext C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unicontext
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## What the binding covers

Deliberately little. A context packet is a document, not a struct: marshalling
one across a C boundary would mean an ownership convention for every string in
it, so packets cross that boundary as JSON through the MCP server instead
(`unicontext serve`). What the ABI does expose is what a caller needs *before*
asking for a packet.

In [1]:
import unicontext

unicontext.version(), unicontext.__version__, unicontext.abi_version()

('1.0.0', '1.0.0', 1)

`__version__` is not a second copy of the version: it is read from
the linked library at import, so it cannot disagree with what is installed.

## The budget bounds

A packet is compiled to a token budget. The accepted range is part of the
contract, and the binding reads it from `UniContext.h` rather than restating
it.

In [2]:
unicontext.BUDGET_MIN, unicontext.BUDGET_MAX

(128, 32768)

`valid_budget` answers for any integer. It does not raise: the C
ABI never lets an exception unwind across the boundary, and the binding keeps
that behaviour rather than inventing an error Python callers would have to
catch.

In [3]:
for tokens in [0, unicontext.BUDGET_MIN - 1, unicontext.BUDGET_MIN,
               4096, unicontext.BUDGET_MAX, unicontext.BUDGET_MAX + 1]:
    print(f"{tokens:>6}  {unicontext.valid_budget(tokens)}")

     0  False
   127  False
   128  True
  4096  True
 32768  True
 32769  False


Screening a budget before the call is the point: a value this
rejects is one `memory_context` would refuse over MCP, and finding that out
locally costs nothing.

In [4]:
requested = 64
budget = min(max(requested, unicontext.BUDGET_MIN), unicontext.BUDGET_MAX)
print(f"asked for {requested}, will ask the server for {budget}")
print("valid:", unicontext.valid_budget(budget))

asked for 64, will ask the server for 128
valid: True


## Where the rest lives

The library itself — indexing, ranking, budgeting, the MCP tools — is Nim, and
the command is the shortest way to it:

```
unicontext index   --manifest /absolute/path/to/unicontext.toml
unicontext serve   --manifest /absolute/path/to/unicontext.toml
```

See `include/UniContext.h` for the C surface, and the book for the full
picture.